# P12B — Audit keselarasan modalitas dan validasi ruang-waktu

Gunakan hasil P12A. Tujuan latihan adalah menguji alur data multimodal yang telah diringkas pada tingkat fitur: inspeksi visual, sensor, lingkungan, lokasi, dan waktu. Kita memeriksa kualitas tiap sumber, lalu membandingkan evaluasi acak dengan pengujian pada aset/waktu yang ditahan. Skor hanya menggambarkan DGP sintesis.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

DATA = Path('Bab12_Paket/Data/bridge_panel_synthetic.csv')
df = pd.read_csv(DATA, parse_dates=['observation_month'])
print('shape:', df.shape, '| bridges:', df.bridge_id.nunique())
print('duplicate bridge-month:', df.duplicated(['bridge_id','observation_month']).sum())
print('month coverage:', df.groupby('bridge_id').observation_month.nunique().describe().to_dict())
print('missing fraction by modality:')
print(df[['vibration_rms','displacement_mm','sensor_coverage_pct','crack_ratio_pct','corrosion_visual_score']].isna().mean().round(3))
assert df.groupby('bridge_id').observation_month.apply(lambda x: x.is_monotonic_increasing).all()


shape: (648, 20) | bridges: 18
duplicate bridge-month: 0
month coverage: {'count': 18.0, 'mean': 36.0, 'std': 0.0, 'min': 36.0, '25%': 36.0, '50%': 36.0, '75%': 36.0, 'max': 36.0}
missing fraction by modality:
vibration_rms             0.083
displacement_mm           0.083
sensor_coverage_pct       0.083
crack_ratio_pct           0.113
corrosion_visual_score    0.113
dtype: float64


In [2]:
# Visual checks: spatial distribution is synthetic and axes are arbitrary km, not a map.
fig, ax = plt.subplots(1,2,figsize=(11,4))
last = df.sort_values('observation_month').groupby('bridge_id').tail(1)
sc=ax[0].scatter(last.easting_km_synth,last.northing_km_synth,c=last.condition_current_synth,cmap='viridis',s=65)
ax[0].set(xlabel='Easting sintetis (km)',ylabel='Northing sintetis (km)',title='Posisi aset sintesis; warna = kondisi terakhir')
fig.colorbar(sc,ax=ax[0],label='Indeks kondisi sintesis')
monthly=df.groupby('observation_month').condition_current_synth.mean()
ax[1].plot(monthly.index,monthly.values,color='#287a73')
ax[1].set(title='Rerata panel per bulan',ylabel='Indeks kondisi sintesis',xlabel='Bulan')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()


## Rancangan fitur dan pemisahan

Indikator visual dan sensor dikumpulkan dekat waktu inspeksi, tetapi tidak selalu serentak. Variabel `photo_offset_days` menunjukkan selisih tanggal foto terhadap bulan observasi; `sensor_coverage_pct` menggambarkan kelengkapan rekaman. Pada praktik nyata, penyatuan perlu menetapkan jendela waktu, aturan agregasi, toleransi keterlambatan, dan perlakuan data yang tidak tersedia. Kita tidak mengisi nilai pada tahap sebelum split: imputasi harus dipelajari hanya pada data latih.

Target adalah `condition_next_3m_synth`. `condition_current_synth` sengaja tidak dimasukkan sebagai prediktor pada eksperimen utama untuk menghindari baseline terlalu mudah sekaligus agar perbandingan antar-sumber terlihat; mahasiswa dapat menguji tambahan fitur itu secara eksplisit sebagai ablation. ID jembatan dan bulan dipakai untuk split/grup, bukan sebagai fitur prediktif langsung.


In [3]:
target='condition_next_3m_synth'
visual=['crack_ratio_pct','corrosion_visual_score','photo_offset_days','inspection_confidence']
sensor=['vibration_rms','displacement_mm','sensor_coverage_pct']
environment=['rain_30d_mm','temperature_C','traffic_index']
context=['easting_km_synth','northing_km_synth','bridge_age_years','span_m','month_index']
num_features=visual+sensor+environment+context
cat_features=['bridge_type']

def make_model(features):
    num=[c for c in features if c in num_features]
    cat=[c for c in features if c in cat_features]
    prep=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
                            ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)],
                           remainder='drop')
    return Pipeline([('prep',prep),('model',RandomForestRegressor(n_estimators=250,min_samples_leaf=3,max_features=.85,random_state=12012,n_jobs=-1))])

def score_split(name, train_idx, test_idx, features):
    model=make_model(features)
    model.fit(df.iloc[train_idx][features],df.iloc[train_idx][target])
    pred=model.predict(df.iloc[test_idx][features]); y=df.iloc[test_idx][target]
    return {'split':name,'n_train':len(train_idx),'n_test':len(test_idx),
            'MAE':mean_absolute_error(y,pred),'RMSE':mean_squared_error(y,pred)**.5,'R2':r2_score(y,pred)}

idx=np.arange(len(df))
tr_rand,te_rand=train_test_split(idx,test_size=.25,random_state=12)
# Hold out assets whole: no bridge appears in both partitions.
held_bridges=set(sorted(df.bridge_id.unique())[-5:])
te_group=np.flatnonzero(df.bridge_id.isin(held_bridges)); tr_group=np.flatnonzero(~df.bridge_id.isin(held_bridges))
# Forecast later months: train months 0..29, test months 30..35 for every bridge.
te_time=np.flatnonzero(df.month_index>=30); tr_time=np.flatnonzero(df.month_index<30)
all_features=visual+sensor+environment+context
results=pd.DataFrame([score_split('random row',tr_rand,te_rand,all_features),
                      score_split('bridge holdout',tr_group,te_group,all_features),
                      score_split('future months',tr_time,te_time,all_features)])
print(results.round(3).to_string(index=False))
print('held-out bridge IDs:',sorted(held_bridges))


         split  n_train  n_test   MAE  RMSE    R2
    random row      486     162 4.149 5.256 0.696
bridge holdout      468     180 5.189 6.720 0.275
 future months      540     108 3.265 4.001 0.787
held-out bridge IDs: ['SYN-14', 'SYN-15', 'SYN-16', 'SYN-17', 'SYN-18']


In [4]:
# Ablation on one fixed bridge holdout: compare source groups under the same split.
feature_sets={'environment+context':environment+context,
              'sensor+environment+context':sensor+environment+context,
              'visual+environment+context':visual+environment+context,
              'all modalities':all_features}
abl=[]
for label,features in feature_sets.items():
    r=score_split(label,tr_group,te_group,features); r['feature_set']=label; abl.append(r)
abl=pd.DataFrame(abl).sort_values('MAE')
print(abl[['feature_set','n_train','n_test','MAE','RMSE','R2']].round(3).to_string(index=False))


               feature_set  n_train  n_test   MAE  RMSE     R2
            all modalities      468     180 5.189 6.720  0.275
sensor+environment+context      468     180 6.210 7.903 -0.003
visual+environment+context      468     180 6.400 8.326 -0.113
       environment+context      468     180 7.768 9.879 -0.567


In [5]:
out=Path('Bab12_Paket/Data'); out.mkdir(parents=True,exist_ok=True)
results.to_csv(out/'P12_split_metrics_synthetic.csv',index=False)
abl.to_csv(out/'P12_modality_ablation_synthetic.csv',index=False)
print('saved metrics to',out)


saved metrics to Bab12_Paket\Data


## Interpretasi dan batas

Split acak biasanya memberi estimasi untuk pengamatan baru yang tercampur dengan kelompok yang sama. Uji bridge holdout lebih dekat ke generalisasi aset yang tidak pernah dilihat; uji future months menilai ekstrapolasi waktu untuk aset yang telah diamati. Keduanya menjawab pertanyaan berbeda. Lima jembatan yang ditahan dipilih berdasarkan ID contoh, bukan dipilih secara acak representatif. Hasil dapat berubah dengan komposisi kelompok dan seed.

Ablation membandingkan ketersediaan kelompok fitur pada satu split; ia bukan bukti kausal bahwa satu modalitas menyebabkan peningkatan. Kinerja dipengaruhi oleh missingness buatan, bentuk DGP, dan korelasi yang ditetapkan generator. Data ini tidak mencakup perubahan domain riil, proses pemeriksaan inspektur, ataupun struktur fisik yang tervalidasi. Gunakan notebook untuk melatih desain analisis, lalu rancang validasi independen sebelum membuat klaim lapangan.
